In [9]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ─── PARÁMETROS GLOBALES ────────────────────────────────────────────────────
TIEMPO_TOTAL     = 8 * 60   # 480 minutos
NUM_CAJEROS      = 3
NUM_REPLICAS     = 10
PROB_RETIRO      = 0.70
PROB_PAGO        = 0.30

# Media de servicio y llegada en minutos
TIPOS_RETIRO = {
    "Rapido":    {"servicio": 1, "llegada": 1, "prob": 0.23},
    "Normal":    {"servicio": 2, "llegada": 2, "prob": 0.40},
    "Lento":     {"servicio": 3, "llegada": 3, "prob": 0.17},
    "MuyLento":  {"servicio": 4, "llegada": 3, "prob": 0.20},
}
TIPOS_PAGO = {
    "Rapido":    {"servicio": 3, "llegada": 1, "prob": 0.10},
    "Normal":    {"servicio": 3, "llegada": 2, "prob": 0.20},
    "Lento":     {"servicio": 5, "llegada": 3, "prob": 0.30},
    "MuyLento":  {"servicio": 7, "llegada": 4, "prob": 0.40},
}

NOMBRES_TIPOS = ["Rapido", "Normal", "Lento", "MuyLento"]
COLORES_TIPOS  = {"Rapido": "#4CAF50", "Normal": "#2196F3",
                  "Lento": "#FF9800",  "MuyLento": "#F44336"}
COLORES_CAJERO = ["#1565C0", "#2E7D32", "#6A1B9A"]


# ─── SIMULACIÓN M/M/1 POR CAJERO ─────────────────────────────────────────────
def simular_cajero(clientes_df, cajero_id, config="mixta"):
    """
    Simula un cajero M/M/1 usando lista de eventos (Event-Driven Simulation).
    Retorna lista de registros con tiempos de espera y servicio.
    """
    # Filtrar clientes según configuración de caja
    if config == "mixta":
        clientes = clientes_df.copy()
    elif config == "solo_retiro":
        clientes = clientes_df[clientes_df["accion"] == "Retiro"].copy()
    elif config == "solo_pago":
        clientes = clientes_df[clientes_df["accion"] == "Pago"].copy()
    else:
        clientes = clientes_df.copy()

    clientes = clientes.sort_values("tiempo_llegada").reset_index(drop=True)

    registros = []
    tiempo_libre_cajero = 0.0   # momento en que el cajero queda libre

    for _, cliente in clientes.iterrows():
        t_llegada  = cliente["tiempo_llegada"]
        t_servicio = cliente["tiempo_servicio"]

        # Tiempo de espera = máx(0, momento libre del cajero - llegada del cliente)
        t_inicio_servicio = max(t_llegada, tiempo_libre_cajero)
        t_espera          = t_inicio_servicio - t_llegada
        t_fin_servicio    = t_inicio_servicio + t_servicio
        t_sistema         = t_espera + t_servicio

        tiempo_libre_cajero = t_fin_servicio

        registros.append({
            "cajero":        cajero_id,
            "accion":        cliente["accion"],
            "tipo_usuario":  cliente["tipo_usuario"],
            "t_llegada":     t_llegada,
            "t_espera":      t_espera,
            "t_servicio":    t_servicio,
            "t_sistema":     t_sistema,
        })

    return pd.DataFrame(registros)


def generar_clientes_global(tiempo_total, seed=None):
    """Genera el flujo global de clientes para los 480 minutos."""
    if seed is not None:
        np.random.seed(seed)



    clientes = []
    tiempo = 0.0
    fin_servicio = 0.0  # reloj del servidor

    while tiempo < tiempo_total:
        # Determinar acción
        if np.random.rand() < PROB_RETIRO:
            accion = "Retiro"
            tipos  = TIPOS_RETIRO
        else:
            accion = "Pago"
            tipos  = TIPOS_PAGO

        # Tipo de usuario
        nombres = list(tipos.keys())
        probs   = [v["prob"] for v in tipos.values()]
        tipo    = np.random.choice(nombres, p=probs)

        # Tiempo entre llegadas
        inter_llegada = np.random.exponential(tipos[tipo]["llegada"])
        tiempo += inter_llegada

        if tiempo >= tiempo_total:
            break

        tiempo_servicio = np.random.exponential(tipos[tipo]["servicio"])

        clientes.append({
            "accion":          accion,
            "tipo_usuario":    tipo,
            "tiempo_llegada":  tiempo,
            "tiempo_servicio": tiempo_servicio,
        })

    return pd.DataFrame(clientes)


def asignar_cajeros(df_clientes, num_cajeros=3):
    """Asigna clientes a cajeros por round-robin (balanceo simple)."""
    df = df_clientes.sort_values("tiempo_llegada").reset_index(drop=True)
    df["cajero_asignado"] = df.index % num_cajeros
    return df


def ejecutar_replica(replica_id, num_cajeros=3, config="mixta"):
    """Ejecuta una réplica completa: genera clientes y simula cada cajero."""
    df_clientes = generar_clientes_global(TIEMPO_TOTAL, seed=None)
    df_asignado = asignar_cajeros(df_clientes, num_cajeros)

    frames = []
    for cajero_id in range(num_cajeros):
        sub = df_asignado[df_asignado["cajero_asignado"] == cajero_id].copy()

        if config == "mixta":
            df_result = simular_cajero(sub, cajero_id, "mixta")
        elif config in ("retiro2pago1", "retiro1pago2"):
            # Configuración especializada
            if config == "retiro2pago1":
                tipo_cajero = "solo_retiro" if cajero_id < 2 else "solo_pago"
            else:
                tipo_cajero = "solo_retiro" if cajero_id == 0 else "solo_pago"
            df_result = simular_cajero(sub, cajero_id, tipo_cajero)
        else:
            df_result = simular_cajero(sub, cajero_id, "mixta")

        frames.append(df_result)

    df_replica = pd.concat(frames, ignore_index=True)
    df_replica["replica"] = replica_id + 1
    return df_replica


def ejecutar_todas_replicas(num_replicas=20, num_cajeros=3, config="mixta"):
    frames = []
    for r in range(num_replicas):
        df = ejecutar_replica(r, num_cajeros, config)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


# ANÁLISIS Y RESOLUCIÓN DE LOS 5 PUNTOS


def calcular_metricas_mm1(df):
    """Calcula métricas teóricas M/M/1 para una subpoblación."""
    n = len(df)
    if n == 0:
        return {}

    lambda_hat = n / TIEMPO_TOTAL          # clientes por minuto
    mu_hat     = 1 / df["t_servicio"].mean()
    rho        = lambda_hat / mu_hat

    resultados = {
        "n_clientes": n,
        "lambda":     round(lambda_hat, 5),
        "mu":         round(mu_hat, 5),
        "rho":        round(rho, 4),
        "estable":    rho < 1,
    }

    if rho < 1:
        resultados.update({
            "L":   round(rho / (1 - rho), 4),
            "Lq":  round(rho**2 / (1 - rho), 4),
            "W":   round(1 / (mu_hat - lambda_hat), 4),
            "Wq":  round(rho / (mu_hat - lambda_hat), 4),
        })
    else:
        resultados.update({"L": np.nan, "Lq": np.nan, "W": np.nan, "Wq": np.nan})

    return resultados


print("=" * 65)
print("  SIMULACIÓN BANCO DE COLOMBIA — M/M/1   (10 réplicas)")
print("=" * 65)

# Ejecutar simulación base (3 cajas mixtas)
df_all = ejecutar_todas_replicas(NUM_REPLICAS, NUM_CAJEROS, "mixta")
print("Réplicas distintas:", df_all["replica"].nunique())

print(f"\nTotal registros simulados: {len(df_all):,}")
print(f"Réplicas: {NUM_REPLICAS} | Cajeros: {NUM_CAJEROS} | Tiempo: {TIEMPO_TOTAL} min\n")


# PUNTO 1: Cajero con menor y mayor tiempo promedio de atención

print("─" * 65)
print("PUNTO 1: Cajero con menor y mayor tiempo promedio de atención")
print("─" * 65)

# Tiempo de atención = tiempo de servicio (lo que dura atender al cliente)
p1 = (df_all.groupby(["replica", "cajero"])["t_servicio"]
      .mean()
      .reset_index()
      .rename(columns={"t_servicio": "t_servicio_prom"}))

# Promedio entre réplicas por cajero
p1_resumen = (p1.groupby("cajero")["t_servicio_prom"]
              .agg(["mean", "std", "min", "max"])
              .reset_index()
              .rename(columns={"mean": "Media", "std": "DE", "min": "Min", "max": "Max"}))
p1_resumen.columns = ["Cajero", "Media(min)", "DE", "Min", "Max"]

# Intervalo de confianza 95%
n = NUM_REPLICAS
t_crit = stats.t.ppf(0.975, df=n-1)
p1_resumen["IC_inf"] = p1_resumen["Media(min)"] - t_crit * p1_resumen["DE"] / np.sqrt(n)
p1_resumen["IC_sup"] = p1_resumen["Media(min)"] + t_crit * p1_resumen["DE"] / np.sqrt(n)

print("\nTiempo promedio de servicio por cajero (promedio de 10 réplicas):\n")
print(p1_resumen.round(4).to_string(index=False))

mejor   = p1_resumen.loc[p1_resumen["Media(min)"].idxmin()]
peor    = p1_resumen.loc[p1_resumen["Media(min)"].idxmax()]
print(f"\n✅ Cajero MÁS RÁPIDO  → Cajero {int(mejor['Cajero'])} con {mejor['Media(min)']:.4f} min promedio")
print(f"⚠️  Cajero MÁS LENTO   → Cajero {int(peor['Cajero'])}  con {peor['Media(min)']:.4f} min promedio")


# PUNTO 2: Promedio de usuarios de cada tipo en la totalidad de cajeros
print("\n" + "─" * 65)
print("PUNTO 2: Promedio de usuarios por tipo en todos los cajeros")
print("─" * 65)

p2 = (df_all.groupby(["replica", "accion", "tipo_usuario"])
      .size()
      .reset_index(name="n_usuarios"))

p2_resumen = (p2.groupby(["accion", "tipo_usuario"])["n_usuarios"]
              .agg(["mean", "std"])
              .reset_index()
              .rename(columns={"mean": "Promedio", "std": "DE"}))
p2_resumen["IC_inf"] = p2_resumen["Promedio"] - t_crit * p2_resumen["DE"] / np.sqrt(n)
p2_resumen["IC_sup"] = p2_resumen["Promedio"] + t_crit * p2_resumen["DE"] / np.sqrt(n)

print("\nPromedio de usuarios por tipo (promedio de las 10 réplicas):\n")
print(p2_resumen.round(2).to_string(index=False))

total_prom = p2_resumen["Promedio"].sum()
print(f"\nTotal promedio de usuarios por réplica: {total_prom:.0f}")


# PUNTO 3: Total de usuarios por tipo en cada réplica + réplica con menos
print("\n" + "─" * 65)
print("PUNTO 3: Total usuarios por tipo en cada réplica")
print("─" * 65)

p3 = (df_all.groupby(["replica", "accion", "tipo_usuario"])
      .size()
      .reset_index(name="n_usuarios"))

# Tabla pivote
p3_pivot = p3.pivot_table(index="replica",
                          columns=["accion", "tipo_usuario"],
                          values="n_usuarios",
                          fill_value=0)
p3_pivot["TOTAL"] = p3_pivot.sum(axis=1)

print("\nTotal de usuarios por réplica y tipo:\n")
print(p3_pivot.to_string())

replica_min = p3_pivot["TOTAL"].idxmin()
print(f"\n📌 Réplica con MENOR cantidad de usuarios: Réplica {replica_min}")
print(f"   Total: {p3_pivot.loc[replica_min, 'TOTAL']} usuarios")
print(f"\nDetalle de la réplica {replica_min}:")
print(p3_pivot.loc[replica_min].to_string())



# PUNTO 4: Necesidad de nuevo cajero (tiempos de espera)

print("\n" + "─" * 65)
print("PUNTO 4: ¿Es necesario un nuevo cajero? (análisis de tiempos de espera)")
print("─" * 65)

# Umbral estándar para banca: Wq > 5 minutos → se recomienda cajero adicional
UMBRAL_ESPERA = 5.0

p4_cajero = (df_all.groupby(["replica", "cajero"])
             .agg(t_espera_prom=("t_espera", "mean"),
                  t_sistema_prom=("t_sistema", "mean"),
                  rho_empirico=("t_servicio", lambda x: x.sum() / TIEMPO_TOTAL))
             .reset_index())

p4_resumen = (p4_cajero.groupby("cajero")
              .agg(Wq_prom=("t_espera_prom", "mean"),
                   Wq_std=("t_espera_prom", "std"),
                   W_prom=("t_sistema_prom", "mean"),
                   rho_prom=("rho_empirico", "mean"))
              .reset_index())

p4_resumen["Wq_IC_inf"] = p4_resumen["Wq_prom"] - t_crit * p4_resumen["Wq_std"] / np.sqrt(n)
p4_resumen["Wq_IC_sup"] = p4_resumen["Wq_prom"] + t_crit * p4_resumen["Wq_std"] / np.sqrt(n)

print("\nMétricas de espera por cajero:\n")
print(p4_resumen.round(4).to_string(index=False))

# Métricas globales (todos los cajeros)
p4_global = (df_all.groupby("replica")
             .agg(Wq=("t_espera", "mean"), W=("t_sistema", "mean"))
             .reset_index())

wq_global  = p4_global["Wq"].mean()
w_global   = p4_global["W"].mean()
ic_wq      = stats.t.interval(0.95, df=n-1, loc=wq_global,
                               scale=stats.sem(p4_global["Wq"]))

print(f"\nTiempo GLOBAL de espera promedio (Wq): {wq_global:.4f} min")
print(f"Tiempo GLOBAL en sistema promedio  (W):  {w_global:.4f} min")
print(f"IC 95% para Wq: [{ic_wq[0]:.4f}, {ic_wq[1]:.4f}] min")

# Métricas teóricas M/M/1 globales
metricas_teoricas = {}
for accion, grp in df_all.groupby("accion"):
    metricas_teoricas[accion] = calcular_metricas_mm1(grp)

print("\nMétricas M/M/1 teóricas por tipo de acción (todas las réplicas):\n")
df_teoricas = pd.DataFrame(metricas_teoricas).T
print(df_teoricas.round(4).to_string())

if wq_global > UMBRAL_ESPERA:
    print(f"\n⚠️  RECOMENDACIÓN: Wq global ({wq_global:.2f} min) > umbral ({UMBRAL_ESPERA} min)")
    print("    → Se recomienda ABRIR UN CAJERO ADICIONAL")
else:
    print(f"\n✅ RECOMENDACIÓN: Wq global ({wq_global:.2f} min) ≤ umbral ({UMBRAL_ESPERA} min)")
    print("    → El sistema opera dentro de parámetros aceptables")
    print("    → No es imprescindible un cajero adicional para tiempos de espera")

# Verificar estabilidad ρ
print("\nFactor de utilización ρ por cajero (debe ser < 1 para estabilidad):")
print(p4_resumen[["cajero", "rho_prom"]].round(4).to_string(index=False))


# PUNTO 5: Configuración óptima (cajas especializadas)

print("\n" + "─" * 65)
print("PUNTO 5: Configuración óptima — cajas especializadas")
print("─" * 65)

configs = {
    "Mixta (3M)":         ("mixta",       3),
    "2 Retiro + 1 Pago":  ("retiro2pago1", 3),
    "1 Retiro + 2 Pago":  ("retiro1pago2", 3),
    "4 Cajas Mixtas":     ("mixta",       4),
}

resultados_configs = {}
for nombre, (config, n_cajeros) in configs.items():
    df_conf = ejecutar_todas_replicas(NUM_REPLICAS, n_cajeros, config)
    wq_m  = df_conf["t_espera"].mean()
    w_m   = df_conf["t_sistema"].mean()
    ts_m  = df_conf["t_servicio"].mean()
    n_m   = len(df_conf) / NUM_REPLICAS
    resultados_configs[nombre] = {
        "Wq (min)": round(wq_m, 4),
        "W (min)":  round(w_m, 4),
        "T_srv (min)": round(ts_m, 4),
        "N prom/dia": round(n_m, 0)
    }
    print(f"  Config '{nombre}' simulada...")

df_configs = pd.DataFrame(resultados_configs).T
print("\nComparación de configuraciones:\n")
print(df_configs.to_string())

mejor_config = df_configs["Wq (min)"].idxmin()
print(f"\n🏆 CONFIGURACIÓN ÓPTIMA: {mejor_config}")
print(f"   → Menor tiempo de espera en cola: {df_configs.loc[mejor_config, 'Wq (min)']} min")

# Análisis de retiro vs pago por separado
df_retiro = df_all[df_all["accion"] == "Retiro"]
df_pago   = df_all[df_all["accion"] == "Pago"]
wq_ret = df_retiro["t_espera"].mean()
wq_pag = df_pago["t_espera"].mean()
w_ret  = df_retiro["t_sistema"].mean()
w_pag  = df_pago["t_sistema"].mean()

print(f"\nAnálisis separado (config. mixta):")
print(f"  Retiros → Wq: {wq_ret:.4f} min | W: {w_ret:.4f} min")
print(f"  Pagos   → Wq: {wq_pag:.4f} min | W: {w_pag:.4f} min")

if wq_pag > wq_ret:
    print("\n  Los pagos generan MAYOR espera → se benefician más de una caja exclusiva")
else:
    print("\n  Los retiros generan MAYOR espera → se benefician más de una caja exclusiva")


# VISUALIZACIONES
print("\n" + "─" * 65)
print("Generando gráficas...")
print("─" * 65)

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
fig.suptitle("Simulación Banco de Colombia — Análisis M/M/1\n(10 réplicas × 3 cajeros × 480 min)",
             fontsize=15, fontweight='bold', y=0.98)

# ── Gráfica 1: Tiempo promedio de servicio por cajero y réplica
ax = axes[0, 0]
for i, cajero in enumerate(range(NUM_CAJEROS)):
    sub = p1[p1["cajero"] == cajero]
    ax.plot(sub["replica"], sub["t_servicio_prom"],
            marker="o", linewidth=2, color=COLORES_CAJERO[i],
            label=f"Cajero {cajero}")
ax.axhline(p1["t_servicio_prom"].mean(), color="gray", linestyle="--", alpha=0.7, label="Global")
ax.set_title("Punto 1: Tiempo Promedio de Servicio por Cajero", fontweight='bold')
ax.set_xlabel("Réplica")
ax.set_ylabel("Minutos")
ax.legend()
ax.set_xticks(range(1, NUM_REPLICAS + 1))
ax.grid(True, alpha=0.3)

# ── Gráfica 2: Distribución del tiempo de servicio (boxplot)
ax = axes[0, 1]
data_box = [p1[p1["cajero"] == c]["t_servicio_prom"].values for c in range(NUM_CAJEROS)]
bp = ax.boxplot(data_box, patch_artist=True, notch=False,
                medianprops=dict(color="white", linewidth=2))
for patch, color in zip(bp['boxes'], COLORES_CAJERO):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_xticklabels([f"Cajero {i}" for i in range(NUM_CAJEROS)])
ax.set_title("Punto 1: Boxplot Tiempo de Servicio por Cajero", fontweight='bold')
ax.set_ylabel("Minutos")
ax.grid(True, alpha=0.3)

# ── Gráfica 3: Promedio de usuarios por tipo (barras agrupadas)
ax = axes[1, 0]
tipos_orden = ["Rapido", "Normal", "Lento", "MuyLento"]
x     = np.arange(len(tipos_orden))
width = 0.35
for j, accion in enumerate(["Retiro", "Pago"]):
    sub = p2_resumen[p2_resumen["accion"] == accion].set_index("tipo_usuario")
    vals = [sub.loc[t, "Promedio"] if t in sub.index else 0 for t in tipos_orden]
    errs = [sub.loc[t, "DE"] / np.sqrt(n) * t_crit if t in sub.index else 0 for t in tipos_orden]
    offset = (j - 0.5) * width
    ax.bar(x + offset, vals, width, label=accion,
           color=["#1565C0", "#C62828"][j], alpha=0.8, yerr=errs, capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(tipos_orden)
ax.set_title("Punto 2: Promedio de Usuarios por Tipo (IC 95%)", fontweight='bold')
ax.set_ylabel("Cantidad de usuarios")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# ── Gráfica 4: Total de usuarios por réplica (barras apiladas)
ax = axes[1, 1]
totales_por_replica = df_all.groupby("replica").size().reset_index(name="total")
colores_bar = plt.cm.viridis(np.linspace(0.2, 0.8, NUM_REPLICAS))
ax.bar(totales_por_replica["replica"], totales_por_replica["total"],
       color=colores_bar, edgecolor='white', linewidth=0.5)
ax.axhline(totales_por_replica["total"].mean(), color="red", linestyle="--",
           linewidth=2, label=f"Promedio: {totales_por_replica['total'].mean():.0f}")
ax.set_title("Punto 3: Total de Usuarios por Réplica", fontweight='bold')
ax.set_xlabel("Réplica")
ax.set_ylabel("Total usuarios")
ax.legend()
ax.set_xticks(range(1, NUM_REPLICAS + 1))
ax.grid(True, alpha=0.3, axis='y')

# ── Gráfica 5: Tiempo de espera en cola por cajero y réplica
ax = axes[2, 0]
for i, cajero in enumerate(range(NUM_CAJEROS)):
    sub = p4_cajero[p4_cajero["cajero"] == cajero]
    ax.plot(sub["replica"], sub["t_espera_prom"],
            marker="s", linewidth=2, color=COLORES_CAJERO[i], label=f"Cajero {cajero}")
ax.axhline(UMBRAL_ESPERA, color="red", linestyle="--", linewidth=2,
           label=f"Umbral {UMBRAL_ESPERA} min")
ax.set_title("Punto 4: Tiempo Promedio de Espera (Wq) por Cajero", fontweight='bold')
ax.set_xlabel("Réplica")
ax.set_ylabel("Minutos de espera")
ax.legend()
ax.set_xticks(range(1, NUM_REPLICAS + 1))
ax.grid(True, alpha=0.3)

# ── Gráfica 6: Comparación de configuraciones
ax = axes[2, 1]
configs_nombres = list(df_configs.index)
wq_vals  = df_configs["Wq (min)"].values
colores_conf = ["#1565C0", "#2E7D32", "#F57F17", "#6A1B9A"]
bars = ax.bar(range(len(configs_nombres)), wq_vals, color=colores_conf, alpha=0.85,
              edgecolor='white', linewidth=1.5)
ax.axhline(UMBRAL_ESPERA, color="red", linestyle="--", linewidth=2,
           label=f"Umbral {UMBRAL_ESPERA} min")
for bar, val in zip(bars, wq_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{val:.3f}", ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xticks(range(len(configs_nombres)))
ax.set_xticklabels(configs_nombres, rotation=15, ha='right', fontsize=9)
ax.set_title("Punto 5: Comparación de Configuraciones (Wq)", fontweight='bold')
ax.set_ylabel("Tiempo de espera en cola (min)")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig("/content/sample_data/graficas_banco.png", dpi=150, bbox_inches='tight')
plt.close()
plt.close()
print("  ✅ graficas_banco.png guardado en sample_data")




# ── Guardar datos tabulares
df_all.to_csv("/content/sample_data/resultados_completos.csv", index=False)
p1_resumen.to_csv("/content/sample_data/punto1_cajeros.csv", index=False)
p2_resumen.to_csv("/content/sample_data/punto2_usuarios.csv", index=False)
p3_pivot.to_csv("/content/sample_data/punto3_replicas.csv")
p4_resumen.to_csv("/content/sample_data/punto4_espera.csv", index=False)
df_configs.to_csv("/content/sample_data/punto5_configs.csv")

print("\n✅ Todos los archivos de datos guardados.")
print("=" * 65)
print("SIMULACIÓN COMPLETADA")
print("=" * 65)

# ── Exportar variables para el documento


  SIMULACIÓN BANCO DE COLOMBIA — M/M/1   (10 réplicas)
Réplicas distintas: 10

Total registros simulados: 2,044
Réplicas: 10 | Cajeros: 3 | Tiempo: 480 min

─────────────────────────────────────────────────────────────────
PUNTO 1: Cajero con menor y mayor tiempo promedio de atención
─────────────────────────────────────────────────────────────────

Tiempo promedio de servicio por cajero (promedio de 10 réplicas):

 Cajero  Media(min)     DE    Min    Max  IC_inf  IC_sup
      0      3.0484 0.3174 2.6306 3.5497  2.8213  3.2755
      1      3.2663 0.4474 2.7040 4.0252  2.9463  3.5863
      2      3.3526 0.4864 2.6485 4.4154  3.0047  3.7005

✅ Cajero MÁS RÁPIDO  → Cajero 0 con 3.0484 min promedio
⚠️  Cajero MÁS LENTO   → Cajero 2  con 3.3526 min promedio

─────────────────────────────────────────────────────────────────
PUNTO 2: Promedio de usuarios por tipo en todos los cajeros
─────────────────────────────────────────────────────────────────

Promedio de usuarios por tipo (promedio de 